In [ ]:
# agriculture_yield_pipeline.ipynb
# ============================================================
# Pour importer sur Colab : Fichier > Importer le notebook
# Ou copiez chaque cellule manuellement.
# Les cellules sont délimitées par # %% (code) ou # %% [markdown]
# ============================================================


# 🌾 Prédiction du Rendement Agricole avec RAPIDS cuML + SHAP
## Pipeline ML accéléré GPU — Google Colab (T4/K80)

**Auteur** : Projet pédagogique  
**Dataset** : FAO FAOSTAT (28 000+ lignes, pays × cultures × années)  
**Modèles** : Random Forest cuML · XGBoost GPU · Régression Linéaire cuML  
**Explicabilité** : SHAP (TreeExplainer + LinearExplainer)  

---
> ⚠️ **Prérequis** : Activer le GPU dans `Modifier > Paramètres du notebook > GPU`



In [ ]:
# ============================================================
# CELLULE 1 : VÉRIFICATION GPU (obligatoire en premier)
# ============================================================
import subprocess
import sys
import os

print("=" * 60)
print("  ÉTAPE 0 : Vérification de l'environnement GPU")
print("=" * 60)

# Vérifier si on est sur Colab
try:
    import google.colab
    ON_COLAB = True
    print("✓ Environnement Google Colab détecté")
except ImportError:
    ON_COLAB = False
    print("ℹ Environnement local détecté (pas Colab)")

# Vérifier le GPU
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        print(f"✓ GPU détecté : {gpu_info}")
        GPU_AVAILABLE = True
    else:
        print("✗ Aucun GPU détecté via nvidia-smi")
        GPU_AVAILABLE = False
except Exception as e:
    print(f"✗ nvidia-smi non disponible : {e}")
    GPU_AVAILABLE = False

if not GPU_AVAILABLE:
    print("\n⚠️  ATTENTION : Aucun GPU disponible !")
    print("   → Sur Colab : Modifier > Paramètres du notebook > GPU")
    print("   → Le code utilisera le CPU comme fallback (plus lent)")

# Afficher la version CUDA
try:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        for line in result.stdout.split('\n'):
            if 'release' in line:
                print(f"✓ CUDA : {line.strip()}")
except:
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=cuda_version',
             '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✓ CUDA version : {result.stdout.strip()}")
    except:
        print("ℹ Version CUDA : non détectable directement")

print()


In [ ]:
# ============================================================
# CELLULE 2 : INSTALLATION RAPIDS (méthode officielle Colab)
# ============================================================
print("=" * 60)
print("  ÉTAPE 1 : Installation de RAPIDS cuML")
print("=" * 60)
print()

RAPIDS_INSTALLED = False

if ON_COLAB and GPU_AVAILABLE:
    print("► Clonage de rapidsai-csp-utils...")
    try:
        # Cloner le repo utilitaire RAPIDS pour Colab
        if not os.path.exists('rapidsai-csp-utils'):
            result = subprocess.run(
                ['git', 'clone', '--quiet',
                 'https://github.com/rapidsai/rapidsai-csp-utils.git'],
                capture_output=True, text=True, timeout=60
            )
            if result.returncode != 0:
                raise RuntimeError(f"git clone échoué : {result.stderr}")
        print("✓ rapidsai-csp-utils cloné")

        print("► Installation de RAPIDS via pip-install.py...")
        print("  (Cette étape peut prendre 5-15 minutes — c'est normal)")
        print()

        # Exécuter le script d'installation officiel
        result = subprocess.run(
            [sys.executable, 'rapidsai-csp-utils/colab/pip-install.py'],
            capture_output=False,  # Afficher la sortie en temps réel
            timeout=1800  # 30 min max
        )

        if result.returncode == 0:
            RAPIDS_INSTALLED = True
            print("\n✓ RAPIDS installé avec succès !")
        else:
            raise RuntimeError("pip-install.py a retourné une erreur")

    except subprocess.TimeoutExpired:
        print("⚠️  Timeout lors de l'installation RAPIDS")
        print("   → Essayez de relancer cette cellule seule")
    except Exception as e:
        print(f"⚠️  Installation RAPIDS échouée : {e}")
        print("   → Fallback vers scikit-learn CPU activé")

    # Installer les dépendances supplémentaires
    print("\n► Installation des dépendances additionnelles...")
    packages = ['xgboost>=2.0', 'shap>=0.45', 'seaborn', 'tqdm']
    for pkg in packages:
        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', pkg],
                capture_output=True, timeout=120
            )
            print(f"  ✓ {pkg}")
        except Exception as e:
            print(f"  ⚠ {pkg} : {e}")

elif not ON_COLAB:
    print("ℹ Environnement local : RAPIDS doit être installé manuellement")
    print("  Voir environment.yml pour conda ou requirements.txt")
    # Vérifier si RAPIDS est déjà installé
    try:
        import cudf, cuml
        RAPIDS_INSTALLED = True
        print("✓ RAPIDS déjà disponible localement")
    except ImportError:
        print("⚠ RAPIDS non trouvé localement → fallback CPU")
else:
    print("⚠ GPU non disponible → installation RAPIDS ignorée")
    print("  Fallback CPU (scikit-learn) sera utilisé")

print(f"\nRAPIDSimétat: {'✓ ACTIVÉ' if RAPIDS_INSTALLED else '⚠ FALLBACK CPU'}")


In [ ]:
# ============================================================
# CELLULE 3 : IMPORTS
# ============================================================
print("=" * 60)
print("  ÉTAPE 2 : Import des bibliothèques")
print("=" * 60)
print()

# ── Bibliothèques standards ────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import requests
import time
import json
from io import StringIO

print("✓ Bibliothèques standards importées")
print(f"  • numpy   {np.__version__}")
print(f"  • pandas  {pd.__version__}")

# ── XGBoost ───────────────────────────────────────────────
try:
    import xgboost as xgb
    print(f"✓ XGBoost {xgb.__version__} importé")
    XGB_OK = True
except ImportError:
    print("⚠ XGBoost non disponible")
    XGB_OK = False

# ── SHAP ──────────────────────────────────────────────────
try:
    import shap
    print(f"✓ SHAP {shap.__version__} importé")
    SHAP_OK = True
except ImportError:
    print("⚠ SHAP non disponible")
    SHAP_OK = False

# ── RAPIDS cuML ───────────────────────────────────────────
USE_GPU = False
cudf = None
cuml_rf = None
cuml_lr = None

try:
    import cudf
    import cuml
    from cuml.ensemble import RandomForestRegressor as cuRF
    from cuml.linear_model import LinearRegression as cuLR
    from cuml.preprocessing import StandardScaler as cuScaler
    print(f"✓ RAPIDS cuML {cuml.__version__} importé")
    USE_GPU = True
    cuml_rf = cuRF
    cuml_lr = cuLR
    GPU_MSG = "🚀 GPU (RAPIDS cuML)"
except ImportError as e:
    print(f"⚠ RAPIDS cuML non disponible ({e})")
    print("  → Fallback : scikit-learn CPU sera utilisé")
    from sklearn.ensemble import RandomForestRegressor as skRF
    from sklearn.linear_model import LinearRegression as skLR
    cuml_rf = skRF
    cuml_lr = skLR
    GPU_MSG = "💻 CPU (scikit-learn fallback)"

print()
print(f"Mode d'exécution : {GPU_MSG}")
print()

# Paramètres matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({
    'figure.dpi': 100,
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
print("✓ Style matplotlib configuré")
print("\n✓ Tous les imports terminés !")


In [ ]:
# ============================================================
# CELLULE 4 : CHARGEMENT DU DATASET FAO
# ============================================================
print("=" * 60)
print("  ÉTAPE 3 : Chargement du dataset FAO FAOSTAT")
print("=" * 60)
print()

def generate_synthetic_fao_dataset(n_rows: int = 28000, seed: int = 42) -> pd.DataFrame:
    """
    Génère un dataset synthétique réaliste mimant la structure FAO FAOSTAT.
    Utilisé comme fallback si le dataset réel n'est pas accessible.
    
    Structure : pays × culture × année avec rendement (hg/ha)
    """
    print("  ► Génération du dataset synthétique FAO (fallback)...")
    rng = np.random.RandomState(seed)

    # Pays représentatifs (6 continents)
    countries = {
        # Asie
        'China': {'region': 'Asia', 'base_yield': 58000, 'std': 8000},
        'India': {'region': 'Asia', 'base_yield': 30000, 'std': 5000},
        'Japan': {'region': 'Asia', 'base_yield': 66000, 'std': 4000},
        'Bangladesh': {'region': 'Asia', 'base_yield': 45000, 'std': 6000},
        'Vietnam': {'region': 'Asia', 'base_yield': 55000, 'std': 5000},
        'Indonesia': {'region': 'Asia', 'base_yield': 51000, 'std': 4500},
        'Thailand': {'region': 'Asia', 'base_yield': 30000, 'std': 4000},
        # Europe
        'France': {'region': 'Europe', 'base_yield': 72000, 'std': 6000},
        'Germany': {'region': 'Europe', 'base_yield': 75000, 'std': 5500},
        'Ukraine': {'region': 'Europe', 'base_yield': 40000, 'std': 8000},
        'Poland': {'region': 'Europe', 'base_yield': 45000, 'std': 6000},
        'Spain': {'region': 'Europe', 'base_yield': 35000, 'std': 7000},
        # Amériques
        'United States': {'region': 'Americas', 'base_yield': 80000, 'std': 5000},
        'Brazil': {'region': 'Americas', 'base_yield': 55000, 'std': 7000},
        'Argentina': {'region': 'Americas', 'base_yield': 65000, 'std': 9000},
        'Canada': {'region': 'Americas', 'base_yield': 35000, 'std': 6000},
        'Mexico': {'region': 'Americas', 'base_yield': 32000, 'std': 5000},
        # Afrique
        'Nigeria': {'region': 'Africa', 'base_yield': 15000, 'std': 4000},
        'Ethiopia': {'region': 'Africa', 'base_yield': 18000, 'std': 4500},
        'Egypt': {'region': 'Africa', 'base_yield': 85000, 'std': 6000},
        'South Africa': {'region': 'Africa', 'base_yield': 48000, 'std': 8000},
        'Kenya': {'region': 'Africa', 'base_yield': 20000, 'std': 5000},
        # Moyen-Orient
        'Turkey': {'region': 'Middle East', 'base_yield': 28000, 'std': 5000},
        'Iran': {'region': 'Middle East', 'base_yield': 38000, 'std': 6000},
        # Océanie
        'Australia': {'region': 'Oceania', 'base_yield': 20000, 'std': 8000},
    }

    # Cultures principales
    crops = {
        'Wheat': {'season': 'Winter', 'water_req': 'Medium', 'base_area': 1500000},
        'Rice, paddy': {'season': 'Summer', 'water_req': 'High', 'base_area': 2000000},
        'Maize (corn)': {'season': 'Summer', 'water_req': 'Medium', 'base_area': 1800000},
        'Soybeans': {'season': 'Summer', 'water_req': 'Low', 'base_area': 1200000},
        'Barley': {'season': 'Winter', 'water_req': 'Low', 'base_area': 800000},
        'Sugar cane': {'season': 'Annual', 'water_req': 'Very High', 'base_area': 500000},
        'Potatoes': {'season': 'Spring', 'water_req': 'Medium', 'base_area': 400000},
        'Cotton lint': {'season': 'Summer', 'water_req': 'High', 'base_area': 600000},
        'Sunflower seed': {'season': 'Summer', 'water_req': 'Low', 'base_area': 700000},
        'Rapeseed': {'season': 'Winter', 'water_req': 'Low', 'base_area': 600000},
    }

    years = list(range(1990, 2023))  # 33 ans

    records = []
    target_n = n_rows

    country_list = list(countries.keys())
    crop_list = list(crops.keys())

    # Tendance technologique globale (+0.8% par an depuis 1990)
    base_year = 1990

    for _ in range(target_n):
        country = rng.choice(country_list)
        crop = rng.choice(crop_list)
        year = rng.choice(years)

        c_info = countries[country]
        cr_info = crops[crop]

        # Tendance temporelle (amélioration progressive)
        trend = 1 + 0.008 * (year - base_year)

        # Facteurs agricoles
        temp_avg = rng.uniform(5, 35)   # °C annuelle
        rainfall = rng.uniform(100, 2500)  # mm/an
        arable_land_pct = rng.uniform(0.05, 0.65)
        fertilizer_kg_ha = rng.uniform(50, 400)
        pesticide_kg_ha = rng.uniform(0.1, 15.0)

        # Calcul du rendement
        base = c_info['base_yield'] * trend
        temp_effect = 1 - 0.0015 * (temp_avg - 20) ** 2
        rain_effect = np.log1p(rainfall) / np.log1p(1000)
        fert_effect = 0.5 + 0.5 * (fertilizer_kg_ha / 400)
        noise = rng.normal(0, c_info['std'])

        yield_val = max(500, base * temp_effect * rain_effect * fert_effect + noise)

        # Surface récoltée
        harvested_area = int(cr_info['base_area'] * rng.uniform(0.5, 1.5))
        production = int(yield_val * harvested_area / 10000)  # tonnes

        records.append({
            'country': country,
            'region': c_info['region'],
            'crop': crop,
            'year': year,
            'season': cr_info['season'],
            'water_requirement': cr_info['water_req'],
            'harvested_area_ha': harvested_area,
            'production_tonnes': production,
            'yield_hg_ha': round(yield_val, 1),  # TARGET
            'temperature_avg_c': round(temp_avg, 1),
            'rainfall_mm': round(rainfall, 1),
            'arable_land_pct': round(arable_land_pct, 4),
            'fertilizer_kg_ha': round(fertilizer_kg_ha, 1),
            'pesticide_kg_ha': round(pesticide_kg_ha, 2),
        })

    df = pd.DataFrame(records)
    print(f"  ✓ Dataset synthétique généré : {len(df):,} lignes × {len(df.columns)} colonnes")
    return df


def load_fao_dataset() -> pd.DataFrame:
    """
    Tente de charger le vrai dataset FAO FAOSTAT.
    En cas d'échec, génère un dataset synthétique réaliste.
    
    URLs testées (données publiques FAO) :
    1. GitHub mirror FAOSTAT crop yields
    2. Kaggle public datasets
    3. Dataset synthétique en fallback
    """
    
    # URL 1 : Kaggle public dataset (crop-yields)
    # Ce dataset est une version nettoyée du FAOSTAT officiel
    urls_to_try = [
        {
            'name': 'Our World in Data - Crop Yields (GitHub)',
            'url': 'https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/Crop%20yields%20-%20FAO%20(2021)/Crop%20yields%20-%20FAO%20(2021).csv',
            'format': 'owid'
        },
        {
            'name': 'FAOSTAT mirror (Hugging Face)',
            'url': 'https://huggingface.co/datasets/nicolaszwickau/faostat-crop-yields/resolve/main/crop_yields.csv',
            'format': 'generic'
        }
    ]

    for source in urls_to_try:
        try:
            print(f"  ► Tentative : {source['name']}")
            print(f"    URL : {source['url'][:80]}...")

            resp = requests.get(source['url'], timeout=30)
            resp.raise_for_status()

            raw_df = pd.read_csv(StringIO(resp.text))
            print(f"    ✓ Téléchargé : {len(raw_df):,} lignes, colonnes = {list(raw_df.columns)[:5]}...")

            # Adapter selon le format
            if source['format'] == 'owid':
                df = adapt_owid_format(raw_df)
            else:
                df = adapt_generic_format(raw_df)

            if df is not None and len(df) > 1000:
                print(f"    ✓ Dataset FAO réel adapté : {len(df):,} lignes")
                return df

        except requests.exceptions.Timeout:
            print(f"    ⚠ Timeout")
        except requests.exceptions.HTTPError as e:
            print(f"    ⚠ HTTP Error : {e}")
        except Exception as e:
            print(f"    ⚠ Erreur : {e}")

    # Fallback : dataset synthétique
    print("\n  ℹ Toutes les URLs ont échoué → génération du dataset synthétique")
    return generate_synthetic_fao_dataset(n_rows=28000)


def adapt_owid_format(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Adapte le format Our World in Data (OWID/FAO) vers notre format standard."""
    try:
        df = raw_df.copy()

        # Renommer les colonnes OWID connues
        rename_map = {}
        for col in df.columns:
            col_lower = col.lower()
            if 'entity' in col_lower or 'country' in col_lower:
                rename_map[col] = 'country'
            elif col_lower == 'year':
                rename_map[col] = 'year'
            elif 'wheat' in col_lower:
                rename_map[col] = 'wheat_yield'
            elif 'rice' in col_lower:
                rename_map[col] = 'rice_yield'
            elif 'maize' in col_lower or 'corn' in col_lower:
                rename_map[col] = 'maize_yield'

        df = df.rename(columns=rename_map)

        if 'country' not in df.columns or 'year' not in df.columns:
            return None

        # Passer au format long (un crop par ligne)
        yield_cols = [c for c in df.columns
                      if any(x in c.lower() for x in ['yield', 'wheat', 'rice', 'maize', 'soy'])]

        if not yield_cols:
            # Essayer de trouver les colonnes numériques
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            yield_cols = [c for c in numeric_cols if c != 'year']

        if not yield_cols:
            return None

        # Melt pour format long
        id_cols = [c for c in ['country', 'year', 'Code'] if c in df.columns]
        long_df = df.melt(
            id_vars=id_cols,
            value_vars=yield_cols[:5],  # Max 5 cultures
            var_name='crop',
            value_name='yield_hg_ha'
        ).dropna(subset=['yield_hg_ha'])

        # Convertir tonnes/ha en hg/ha (×10000) si nécessaire
        if long_df['yield_hg_ha'].median() < 100:
            long_df['yield_hg_ha'] = long_df['yield_hg_ha'] * 10000

        # Ajouter features dérivées synthétiques (pour l'exercice)
        rng = np.random.RandomState(42)
        n = len(long_df)
        long_df['region'] = 'Unknown'
        long_df['season'] = 'Annual'
        long_df['harvested_area_ha'] = rng.randint(100000, 5000000, n)
        long_df['temperature_avg_c'] = rng.uniform(5, 35, n).round(1)
        long_df['rainfall_mm'] = rng.uniform(100, 2500, n).round(1)
        long_df['arable_land_pct'] = rng.uniform(0.05, 0.65, n).round(4)
        long_df['fertilizer_kg_ha'] = rng.uniform(50, 400, n).round(1)
        long_df['pesticide_kg_ha'] = rng.uniform(0.1, 15, n).round(2)

        long_df = long_df[long_df['yield_hg_ha'] > 0].reset_index(drop=True)
        return long_df

    except Exception as e:
        print(f"    ⚠ adapt_owid_format échoué : {e}")
        return None


def adapt_generic_format(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Adapte un format générique vers notre format standard."""
    try:
        df = raw_df.copy()
        df.columns = [c.lower().replace(' ', '_').replace('-', '_') for c in df.columns]

        # Chercher la colonne cible
        target_candidates = ['yield_hg_ha', 'yield', 'value', 'hg_ha', 'production']
        target_col = None
        for cand in target_candidates:
            if cand in df.columns:
                target_col = cand
                break

        if target_col is None:
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                target_col = numeric_cols[-1]
            else:
                return None

        if target_col != 'yield_hg_ha':
            df = df.rename(columns={target_col: 'yield_hg_ha'})

        return df if len(df) > 1000 else None

    except Exception as e:
        print(f"    ⚠ adapt_generic_format échoué : {e}")
        return None


# ── Chargement ─────────────────────────────────────────────
print("► Chargement du dataset FAO...")
t0 = time.time()
df_raw = load_fao_dataset()
elapsed = time.time() - t0

print(f"\n✓ Dataset chargé en {elapsed:.1f}s")
print(f"  Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")
print(f"  Colonnes   : {list(df_raw.columns)}")
print(f"\nAperçu :")
df_raw.head(3)


In [ ]:
# ============================================================
# CELLULE 5 : ANALYSE EXPLORATOIRE (EDA)
# ============================================================
print("=" * 60)
print("  ÉTAPE 4 : Analyse Exploratoire des Données (EDA)")
print("=" * 60)
print()

# ── Statistiques de base ───────────────────────────────────
print("► Statistiques descriptives :")
print(df_raw.describe().round(1).to_string())
print()

# Valeurs manquantes
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage %': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0]
if len(missing_df) > 0:
    print("► Valeurs manquantes :")
    print(missing_df.to_string())
else:
    print("✓ Aucune valeur manquante détectée")
print()

# ── Visualisations EDA ─────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Analyse Exploratoire – Dataset FAO Rendements Agricoles',
             fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Distribution du rendement
ax1 = fig.add_subplot(gs[0, 0])
yield_data = df_raw['yield_hg_ha'].dropna()
ax1.hist(yield_data, bins=50, color='#2196F3', alpha=0.7, edgecolor='white')
ax1.axvline(yield_data.mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'Moy: {yield_data.mean():,.0f}')
ax1.axvline(yield_data.median(), color='orange', linestyle='--', linewidth=1.5,
            label=f'Méd: {yield_data.median():,.0f}')
ax1.set_title('Distribution du Rendement (hg/ha)')
ax1.set_xlabel('Rendement (hg/ha)')
ax1.set_ylabel('Fréquence')
ax1.legend(fontsize=9)

# 2. Évolution temporelle si colonne year existe
ax2 = fig.add_subplot(gs[0, 1])
if 'year' in df_raw.columns:
    yearly = df_raw.groupby('year')['yield_hg_ha'].mean()
    ax2.plot(yearly.index, yearly.values, color='#4CAF50', linewidth=2, marker='o',
             markersize=3)
    ax2.fill_between(yearly.index, yearly.values, alpha=0.2, color='#4CAF50')
    ax2.set_title('Évolution du Rendement Moyen par Année')
    ax2.set_xlabel('Année')
    ax2.set_ylabel('Rendement moyen (hg/ha)')
else:
    ax2.text(0.5, 0.5, 'Colonne "year"\nnon disponible',
             ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Évolution temporelle')

# 3. Top rendements par culture (si colonne crop)
ax3 = fig.add_subplot(gs[0, 2])
if 'crop' in df_raw.columns:
    crop_yields = df_raw.groupby('crop')['yield_hg_ha'].mean().sort_values(ascending=True)
    top_crops = crop_yields.tail(8)
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_crops)))
    bars = ax3.barh(range(len(top_crops)), top_crops.values, color=colors)
    ax3.set_yticks(range(len(top_crops)))
    ax3.set_yticklabels([c[:20] for c in top_crops.index], fontsize=8)
    ax3.set_title('Rendement Moyen par Culture')
    ax3.set_xlabel('Rendement moyen (hg/ha)')
else:
    ax3.text(0.5, 0.5, 'Colonne "crop"\nnon disponible',
             ha='center', va='center', transform=ax3.transAxes)

# 4. Heatmap corrélations
ax4 = fig.add_subplot(gs[1, :2])
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if df_raw[c].std() > 0]
if len(numeric_cols) >= 3:
    corr_cols = numeric_cols[:10]  # Max 10 features
    corr_matrix = df_raw[corr_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5,
                annot_kws={'size': 8}, ax=ax4,
                cbar_kws={'shrink': 0.8})
    ax4.set_title('Matrice de Corrélation des Features Numériques')
    ax4.tick_params(axis='both', labelsize=8)

# 5. Boxplot par région (si disponible)
ax5 = fig.add_subplot(gs[1, 2])
if 'region' in df_raw.columns and df_raw['region'].nunique() > 1:
    regions = df_raw['region'].value_counts().head(6).index
    region_data = [df_raw[df_raw['region'] == r]['yield_hg_ha'].dropna().values
                   for r in regions]
    bp = ax5.boxplot(region_data, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2})
    colors_box = plt.cm.Set3(np.linspace(0, 1, len(regions)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    ax5.set_xticklabels([r[:10] for r in regions], rotation=30, ha='right', fontsize=8)
    ax5.set_title('Distribution par Région')
    ax5.set_ylabel('Rendement (hg/ha)')
else:
    numeric_sample = df_raw[numeric_cols[:5]].sample(min(500, len(df_raw)))
    ax5.boxplot([numeric_sample[c].dropna() for c in numeric_cols[:5]],
                patch_artist=True)
    ax5.set_xticklabels([c[:12] for c in numeric_cols[:5]], rotation=30, ha='right',
                        fontsize=8)
    ax5.set_title('Boxplot Features Numériques')

plt.savefig('eda_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Graphiques EDA sauvegardés (eda_analysis.png)")


In [ ]:
# ============================================================
# CELLULE 6 : PRÉTRAITEMENT DES DONNÉES
# ============================================================
print("=" * 60)
print("  ÉTAPE 5 : Prétraitement des Données")
print("=" * 60)
print()

df = df_raw.copy()

# ── 5.1 Suppression des lignes sans target ─────────────────
before = len(df)
df = df.dropna(subset=['yield_hg_ha'])
df = df[df['yield_hg_ha'] > 0]
print(f"✓ Suppression lignes sans yield : {before - len(df)} lignes retirées")

# ── 5.2 Identification des colonnes ────────────────────────
TARGET = 'yield_hg_ha'
EXCLUDE_COLS = ['yield_hg_ha', 'production_tonnes']  # Leakage potentiel

# Colonnes catégorielles
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in EXCLUDE_COLS]

# Colonnes numériques (sans target ni exclues)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in EXCLUDE_COLS + [TARGET]]

print(f"✓ Target : {TARGET}")
print(f"✓ Features catégorielles ({len(cat_cols)}) : {cat_cols}")
print(f"✓ Features numériques ({len(num_cols)}) : {num_cols}")

# ── 5.3 Imputation des valeurs manquantes ──────────────────
print("\n► Imputation des valeurs manquantes...")

# Numériques → médiane (robuste aux outliers)
for col in num_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  ✓ {col}: {n_missing} NaN → médiane ({median_val:.2f})")

# Catégorielles → mode
for col in cat_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  ✓ {col}: {n_missing} NaN → mode ('{mode_val}')")

print("✓ Imputation terminée")

# ── 5.4 Encodage Label des variables catégorielles ─────────
print("\n► Encodage LabelEncoder des variables catégorielles...")
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    n_classes = len(le.classes_)
    print(f"  ✓ {col}: {n_classes} classes → entiers 0-{n_classes-1}")

# ── 5.5 Feature Engineering ────────────────────────────────
print("\n► Feature Engineering...")

# Log transform du rendement (distribution plus normale)
df['log_yield'] = np.log1p(df[TARGET])

# Interaction features si les colonnes existent
if 'fertilizer_kg_ha' in num_cols and 'rainfall_mm' in num_cols:
    df['fert_rain_ratio'] = df['fertilizer_kg_ha'] / (df['rainfall_mm'] + 1)
    num_cols.append('fert_rain_ratio')
    print("  ✓ Feature ajoutée : fert_rain_ratio = fertilizer/rainfall")

if 'temperature_avg_c' in num_cols and 'rainfall_mm' in num_cols:
    df['heat_moisture_idx'] = df['temperature_avg_c'] * np.log1p(df['rainfall_mm'])
    num_cols.append('heat_moisture_idx')
    print("  ✓ Feature ajoutée : heat_moisture_idx = temp × log(rain)")

# ── 5.6 Sélection finale des features ─────────────────────
FEATURE_COLS = cat_cols + num_cols
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f"\n✓ Features finales ({len(FEATURE_COLS)}) : {FEATURE_COLS}")

# ── 5.7 Assertion qualité ──────────────────────────────────
assert df[FEATURE_COLS].isnull().sum().sum() == 0, \
    "ERREUR : Des NaN subsistent dans les features !"
assert df[TARGET].isnull().sum() == 0, \
    "ERREUR : Des NaN subsistent dans la target !"
assert len(df) > 1000, \
    f"ERREUR : Dataset trop petit ({len(df)} lignes)"

print(f"\n✓ Assertions qualité : OK")
print(f"✓ Dataset final : {len(df):,} lignes × {len(FEATURE_COLS)} features")
print(f"  Target (yield_hg_ha) : min={df[TARGET].min():.0f}, "
      f"mean={df[TARGET].mean():.0f}, max={df[TARGET].max():.0f}")


In [ ]:
# ============================================================
# CELLULE 7 : SPLIT TRAIN/TEST + CONVERSION GPU
# ============================================================
print("=" * 60)
print("  ÉTAPE 6 : Split Train/Test + Conversion GPU")
print("=" * 60)
print()

# ── 7.1 Préparation des matrices ───────────────────────────
X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

print(f"✓ Matrices X : {X.shape}, dtype = {X.dtype}")
print(f"✓ Vecteur y : {y.shape}, dtype = {y.dtype}")

# ── 7.2 Split 80/20 ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print(f"\n✓ Train : {X_train.shape[0]:,} échantillons ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"✓ Test  : {X_test.shape[0]:,} échantillons ({X_test.shape[0]/len(df)*100:.0f}%)")

# ── 7.3 Normalisation (pour Régression Linéaire) ───────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)
print("\n✓ StandardScaler ajusté sur train, appliqué sur test")

# ── 7.4 Conversion en DataFrames GPU (cuDF) ────────────────
if USE_GPU:
    print("\n► Conversion NumPy → cuDF (GPU Memory)...")
    try:
        X_train_cu = cudf.DataFrame(X_train, columns=FEATURE_COLS)
        X_test_cu = cudf.DataFrame(X_test, columns=FEATURE_COLS)
        y_train_cu = cudf.Series(y_train)
        y_test_cu = cudf.Series(y_test)

        X_train_scaled_cu = cudf.DataFrame(X_train_scaled, columns=FEATURE_COLS)
        X_test_scaled_cu = cudf.DataFrame(X_test_scaled, columns=FEATURE_COLS)

        print(f"  ✓ X_train_cu : {X_train_cu.shape} (GPU)")
        print(f"  ✓ X_test_cu  : {X_test_cu.shape} (GPU)")
        print(f"  ✓ y_train_cu : {len(y_train_cu):,} valeurs (GPU)")

        # Assertion GPU : vérifier que les données sont bien en mémoire GPU
        assert hasattr(X_train_cu, '_column'), "X_train_cu n'est pas un cuDF DataFrame !"
        print("\n✓ CONFIRMATION GPU : Données chargées en mémoire VRAM")

    except Exception as e:
        print(f"  ⚠ Conversion cuDF échouée : {e}")
        print("  → Fallback vers NumPy CPU")
        USE_GPU = False
        X_train_cu, X_test_cu = X_train, X_test
        y_train_cu, y_test_cu = y_train, y_test
        X_train_scaled_cu = X_train_scaled
        X_test_scaled_cu = X_test_scaled
else:
    # Mode CPU : garder NumPy
    X_train_cu, X_test_cu = X_train, X_test
    y_train_cu, y_test_cu = y_train, y_test
    X_train_scaled_cu = X_train_scaled
    X_test_scaled_cu = X_test_scaled
    print("ℹ Mode CPU : données NumPy conservées")


In [ ]:
# ============================================================
# CELLULE 8 : ENTRAÎNEMENT DES MODÈLES
# ============================================================
print("=" * 60)
print(f"  ÉTAPE 7 : Entraînement des Modèles ({GPU_MSG})")
print("=" * 60)
print()

results = {}  # Stocke les métriques

# ──────────────────────────────────────────────────────────
# MODÈLE 1 : Random Forest (cuML ou sklearn)
# ──────────────────────────────────────────────────────────
print("━" * 50)
print(f"► MODÈLE 1 : Random Forest ({GPU_MSG})")
print("━" * 50)

t0 = time.time()

try:
    if USE_GPU:
        # cuML Random Forest – paramètres optimisés pour T4
        rf_model = cuml_rf(
            n_estimators=100,
            max_depth=16,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=0.8,       # 80% des features par split
            n_bins=32,              # Bins pour GPU (spécifique cuML)
            n_streams=4,            # Parallélisme GPU
            random_state=42,
            verbose=0
        )
    else:
        # scikit-learn fallback
        rf_model = cuml_rf(
            n_estimators=100,
            max_depth=16,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=0.8,
            n_jobs=-1,
            random_state=42,
            verbose=0
        )

    rf_model.fit(X_train_cu, y_train_cu)
    train_time_rf = time.time() - t0

    # Prédictions
    rf_pred_train = np.array(rf_model.predict(X_train_cu))
    rf_pred_test = np.array(rf_model.predict(X_test_cu))

    # Métriques
    rf_rmse_train = np.sqrt(mean_squared_error(y_train, rf_pred_train))
    rf_r2_train = r2_score(y_train, rf_pred_train)
    rf_rmse_test = np.sqrt(mean_squared_error(y_test, rf_pred_test))
    rf_r2_test = r2_score(y_test, rf_pred_test)

    results['Random Forest'] = {
        'model': rf_model,
        'pred_test': rf_pred_test,
        'pred_train': rf_pred_train,
        'rmse_train': rf_rmse_train,
        'r2_train': rf_r2_train,
        'rmse_test': rf_rmse_test,
        'r2_test': rf_r2_test,
        'train_time': train_time_rf,
        'color': '#2196F3',
        'trained': True
    }

    print(f"  ✓ Entraîné en {train_time_rf:.1f}s")
    print(f"  Train → RMSE: {rf_rmse_train:>10,.1f} | R²: {rf_r2_train:.4f}")
    print(f"  Test  → RMSE: {rf_rmse_test:>10,.1f} | R²: {rf_r2_test:.4f}")
    RF_OK = True

except Exception as e:
    print(f"  ✗ Random Forest échoué : {e}")
    RF_OK = False
    results['Random Forest'] = {'trained': False, 'color': '#2196F3'}

# ──────────────────────────────────────────────────────────
# MODÈLE 2 : XGBoost (GPU)
# ──────────────────────────────────────────────────────────
print()
print("━" * 50)
print("► MODÈLE 2 : XGBoost GPU")
print("━" * 50)

t0 = time.time()

try:
    if not XGB_OK:
        raise ImportError("XGBoost non installé")

    # Paramètres XGBoost avec GPU
    xgb_params = {
        'n_estimators': 300,
        'max_depth': 8,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_alpha': 0.1,       # L1 regularization
        'reg_lambda': 1.0,      # L2 regularization
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0,
        'eval_metric': 'rmse',
    }

    # Activer GPU si disponible
    if GPU_AVAILABLE:
        xgb_params['device'] = 'cuda'
        xgb_params['tree_method'] = 'hist'
        print("  (device=cuda activé)")
    else:
        xgb_params['tree_method'] = 'hist'
        print("  (CPU mode)")

    xgb_model = xgb.XGBRegressor(**xgb_params)

    # Entraînement avec early stopping sur validation
    xgb_model.fit(
        X_train, y_train,  # XGBoost préfère NumPy/pandas
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    train_time_xgb = time.time() - t0

    # Prédictions
    xgb_pred_train = xgb_model.predict(X_train)
    xgb_pred_test = xgb_model.predict(X_test)

    # Métriques
    xgb_rmse_train = np.sqrt(mean_squared_error(y_train, xgb_pred_train))
    xgb_r2_train = r2_score(y_train, xgb_pred_train)
    xgb_rmse_test = np.sqrt(mean_squared_error(y_test, xgb_pred_test))
    xgb_r2_test = r2_score(y_test, xgb_pred_test)

    results['XGBoost'] = {
        'model': xgb_model,
        'pred_test': xgb_pred_test,
        'pred_train': xgb_pred_train,
        'rmse_train': xgb_rmse_train,
        'r2_train': xgb_r2_train,
        'rmse_test': xgb_rmse_test,
        'r2_test': xgb_r2_test,
        'train_time': train_time_xgb,
        'color': '#FF9800',
        'trained': True
    }

    print(f"  ✓ Entraîné en {train_time_xgb:.1f}s")
    print(f"  Train → RMSE: {xgb_rmse_train:>10,.1f} | R²: {xgb_r2_train:.4f}")
    print(f"  Test  → RMSE: {xgb_rmse_test:>10,.1f} | R²: {xgb_r2_test:.4f}")
    XGB_MODEL_OK = True

except Exception as e:
    print(f"  ✗ XGBoost échoué : {e}")
    XGB_MODEL_OK = False
    results['XGBoost'] = {'trained': False, 'color': '#FF9800'}

# ──────────────────────────────────────────────────────────
# MODÈLE 3 : Régression Linéaire (cuML ou sklearn)
# ──────────────────────────────────────────────────────────
print()
print("━" * 50)
print(f"► MODÈLE 3 : Régression Linéaire ({GPU_MSG})")
print("━" * 50)

t0 = time.time()

try:
    if USE_GPU:
        lr_model = cuml_lr(
            fit_intercept=True,
            normalize=False,   # Déprécié dans versions récentes mais conservé
            algorithm='eig',   # Plus stable que 'svd' sur GPU
            verbose=0
        )
    else:
        lr_model = cuml_lr(
            fit_intercept=True,
            n_jobs=-1
        )

    lr_model.fit(X_train_scaled_cu, y_train_cu)
    train_time_lr = time.time() - t0

    # Prédictions (renormaliser au besoin)
    lr_pred_train = np.array(lr_model.predict(X_train_scaled_cu))
    lr_pred_test = np.array(lr_model.predict(X_test_scaled_cu))

    # Clip pour éviter les prédictions négatives
    lr_pred_train = np.clip(lr_pred_train, 0, None)
    lr_pred_test = np.clip(lr_pred_test, 0, None)

    # Métriques
    lr_rmse_train = np.sqrt(mean_squared_error(y_train, lr_pred_train))
    lr_r2_train = r2_score(y_train, lr_pred_train)
    lr_rmse_test = np.sqrt(mean_squared_error(y_test, lr_pred_test))
    lr_r2_test = r2_score(y_test, lr_pred_test)

    results['Lin. Regression'] = {
        'model': lr_model,
        'pred_test': lr_pred_test,
        'pred_train': lr_pred_train,
        'rmse_train': lr_rmse_train,
        'r2_train': lr_r2_train,
        'rmse_test': lr_rmse_test,
        'r2_test': lr_r2_test,
        'train_time': train_time_lr,
        'color': '#4CAF50',
        'trained': True
    }

    print(f"  ✓ Entraîné en {train_time_lr:.1f}s")
    print(f"  Train → RMSE: {lr_rmse_train:>10,.1f} | R²: {lr_r2_train:.4f}")
    print(f"  Test  → RMSE: {lr_rmse_test:>10,.1f} | R²: {lr_r2_test:.4f}")
    LR_OK = True

except Exception as e:
    print(f"  ✗ Régression Linéaire échouée : {e}")
    LR_OK = False
    results['Lin. Regression'] = {'trained': False, 'color': '#4CAF50'}

print()
print("=" * 50)
print("✓ ENTRAÎNEMENT TERMINÉ")
print("=" * 50)


In [ ]:
# ============================================================
# CELLULE 9 : COMPARAISON DES MODÈLES
# ============================================================
print("=" * 60)
print("  ÉTAPE 8 : Comparaison et Visualisation des Résultats")
print("=" * 60)
print()

trained_models = {k: v for k, v in results.items() if v.get('trained', False)}

# ── Tableau récapitulatif ──────────────────────────────────
print(f"{'Modèle':<20} {'RMSE Train':>12} {'R² Train':>10} {'RMSE Test':>12} {'R² Test':>10} {'Temps':>8}")
print("─" * 78)

for name, res in trained_models.items():
    print(f"{name:<20} {res['rmse_train']:>12,.1f} {res['r2_train']:>10.4f} "
          f"{res['rmse_test']:>12,.1f} {res['r2_test']:>10.4f} "
          f"{res['train_time']:>7.1f}s")

best_model_name = min(trained_models, key=lambda k: trained_models[k]['rmse_test'])
print(f"\n🏆 Meilleur modèle (RMSE test) : {best_model_name}")

# ── Graphiques comparatifs ─────────────────────────────────
n_models = len(trained_models)
if n_models == 0:
    print("⚠ Aucun modèle entraîné avec succès")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f'Comparaison des Modèles ML — {GPU_MSG}',
                 fontsize=14, fontweight='bold')

    model_names = list(trained_models.keys())
    colors = [trained_models[m]['color'] for m in model_names]

    # ── Plot 1 : RMSE comparaison ──────────────────────────
    ax = axes[0, 0]
    x = np.arange(len(model_names))
    width = 0.35
    bars1 = ax.bar(x - width/2, [trained_models[m]['rmse_train'] for m in model_names],
                   width, label='Train', color=colors, alpha=0.9)
    bars2 = ax.bar(x + width/2, [trained_models[m]['rmse_test'] for m in model_names],
                   width, label='Test', color=colors, alpha=0.5, edgecolor='black',
                   linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.set_ylabel('RMSE (hg/ha)')
    ax.set_title('RMSE — Train vs Test')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'{v:,.0f}'))
    for bar in bars2:
        h = bar.get_height()
        ax.annotate(f'{h:,.0f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

    # ── Plot 2 : R² comparaison ────────────────────────────
    ax = axes[0, 1]
    bars_r2 = ax.bar(model_names,
                     [trained_models[m]['r2_test'] for m in model_names],
                     color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.axhline(y=0.9, color='green', linestyle='--', linewidth=1, label='R²=0.9 (Excellent)')
    ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1, label='R²=0.7 (Bon)')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('R² Score')
    ax.set_title('R² Score (Test Set)')
    ax.legend(fontsize=8)
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    for bar, name in zip(bars_r2, model_names):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # ── Plot 3 : Temps d'entraînement ─────────────────────
    ax = axes[0, 2]
    times = [trained_models[m]['train_time'] for m in model_names]
    bars_t = ax.barh(model_names, times, color=colors, alpha=0.85)
    ax.set_xlabel('Temps (secondes)')
    ax.set_title(f"Temps d'Entraînement ({GPU_MSG})")
    for bar, t in zip(bars_t, times):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{t:.1f}s', va='center', fontsize=10)

    # ── Plots 4-6 : Prédictions vs Réalité ────────────────
    for idx, (name, res) in enumerate(list(trained_models.items())[:3]):
        ax = axes[1, idx]
        y_true = y_test
        y_pred = res['pred_test']

        # Scatter plot (échantillon pour lisibilité)
        n_sample = min(1000, len(y_true))
        idx_sample = np.random.choice(len(y_true), n_sample, replace=False)
        ax.scatter(y_true[idx_sample], y_pred[idx_sample],
                   alpha=0.4, s=8, color=res['color'], label='Prédictions')

        # Ligne parfaite
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--',
                linewidth=1.5, label='Prédiction parfaite')

        ax.set_xlabel('Valeur Réelle (hg/ha)')
        ax.set_ylabel('Valeur Prédite (hg/ha)')
        ax.set_title(f'{name}\nR²={res["r2_test"]:.4f} | RMSE={res["rmse_test"]:,.0f}')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'{v/1000:.0f}k'))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'{v/1000:.0f}k'))

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Graphique comparaison sauvegardé (model_comparison.png)")


In [ ]:
# ============================================================
# CELLULE 10 : ANALYSE SHAP
# ============================================================
print("=" * 60)
print("  ÉTAPE 9 : Analyse SHAP — Explicabilité des Modèles")
print("=" * 60)
print()

if not SHAP_OK:
    print("⚠ SHAP non disponible → cellule ignorée")
else:
    # Échantillon pour SHAP (évite les problèmes de mémoire)
    N_SHAP = min(500, len(X_test))
    idx_shap = np.random.choice(len(X_test), N_SHAP, replace=False)
    X_shap_np = X_test[idx_shap]  # numpy array

    shap_results = {}

    # ── SHAP pour Random Forest ────────────────────────────
    if RF_OK:
        print("► Calcul SHAP pour Random Forest...")
        try:
            # cuML RF → TreeExplainer (après conversion vers sklearn-like)
            # Pour cuML, on utilise un wrapper ou on extrait via sklearn
            rf_explainer = shap.TreeExplainer(
                results['Random Forest']['model'],
                feature_perturbation='tree_path_dependent',
                model_output='raw'
            )
            shap_values_rf = rf_explainer.shap_values(X_shap_np)
            shap_results['Random Forest'] = {
                'explainer': rf_explainer,
                'values': shap_values_rf,
                'color': '#2196F3'
            }
            print(f"  ✓ SHAP RF calculé : {shap_values_rf.shape}")
        except Exception as e:
            print(f"  ⚠ SHAP RF TreeExplainer échoué ({e})")
            print("  → Tentative avec KernelExplainer (plus lent)...")
            try:
                # KernelExplainer comme fallback universel
                N_background = min(100, len(X_train))
                background = shap.sample(X_train, N_background)
                rf_explainer = shap.KernelExplainer(
                    results['Random Forest']['model'].predict,
                    background
                )
                shap_values_rf = rf_explainer.shap_values(
                    X_shap_np[:100]  # Réduit pour KernelExplainer (lent)
                )
                shap_results['Random Forest'] = {
                    'explainer': rf_explainer,
                    'values': shap_values_rf,
                    'color': '#2196F3'
                }
                X_shap_rf = X_shap_np[:100]
                print(f"  ✓ SHAP RF KernelExplainer calculé")
            except Exception as e2:
                print(f"  ✗ SHAP RF totalement échoué : {e2}")

    # ── SHAP pour XGBoost ──────────────────────────────────
    if XGB_MODEL_OK:
        print("\n► Calcul SHAP pour XGBoost...")
        try:
            xgb_explainer = shap.TreeExplainer(results['XGBoost']['model'])
            shap_values_xgb = xgb_explainer.shap_values(X_shap_np)
            shap_results['XGBoost'] = {
                'explainer': xgb_explainer,
                'values': shap_values_xgb,
                'color': '#FF9800'
            }
            print(f"  ✓ SHAP XGBoost calculé : {shap_values_xgb.shape}")
        except Exception as e:
            print(f"  ✗ SHAP XGBoost échoué : {e}")

    # ── SHAP pour Régression Linéaire ─────────────────────
    if LR_OK:
        print("\n► Calcul SHAP pour Régression Linéaire...")
        try:
            # LinearExplainer nécessite les données normalisées
            background_scaled = shap.sample(X_train_scaled, min(100, len(X_train_scaled)))
            lr_explainer = shap.LinearExplainer(
                results['Lin. Regression']['model'],
                background_scaled,
                feature_perturbation='correlation_dependent'
            )
            X_shap_scaled = X_test_scaled[idx_shap]
            shap_values_lr = lr_explainer.shap_values(X_shap_scaled)
            shap_results['Lin. Regression'] = {
                'explainer': lr_explainer,
                'values': shap_values_lr,
                'color': '#4CAF50'
            }
            print(f"  ✓ SHAP Linear calculé : {shap_values_lr.shape}")
        except Exception as e:
            print(f"  ⚠ SHAP Linear échoué ({e}), tentative KernelExplainer...")
            try:
                background_scaled = shap.sample(X_train_scaled, 50)
                lr_explainer = shap.KernelExplainer(
                    results['Lin. Regression']['model'].predict,
                    background_scaled
                )
                shap_values_lr = lr_explainer.shap_values(X_test_scaled[:50])
                shap_results['Lin. Regression'] = {
                    'explainer': lr_explainer,
                    'values': shap_values_lr,
                    'color': '#4CAF50'
                }
                print(f"  ✓ SHAP Linear KernelExplainer calculé")
            except Exception as e2:
                print(f"  ✗ SHAP Linear totalement échoué : {e2}")

    print(f"\n✓ SHAP calculé pour {len(shap_results)} modèle(s) : {list(shap_results.keys())}")


In [ ]:
# ============================================================
# CELLULE 11 : VISUALISATIONS SHAP
# ============================================================
print("=" * 60)
print("  ÉTAPE 10 : Visualisations SHAP")
print("=" * 60)
print()

if not SHAP_OK or len(shap_results) == 0:
    print("⚠ Aucun résultat SHAP disponible pour visualisation")
else:
    # Noms des features (raccourcis pour l'affichage)
    feature_names_display = [f[:25] for f in FEATURE_COLS]

    for model_name, shap_data in shap_results.items():
        print(f"\n{'─'*50}")
        print(f"  Visualisations SHAP : {model_name}")
        print(f"{'─'*50}")

        sv = shap_data['values']

        # Vérifier que sv est 2D
        if isinstance(sv, list):
            sv = sv[0] if len(sv) > 0 else None
        if sv is None:
            print("  ⚠ Valeurs SHAP invalides")
            continue

        sv = np.array(sv)
        if sv.ndim == 1:
            sv = sv.reshape(-1, 1)

        # Adapter X_shap selon si c'est normalisé ou non
        X_for_plot = X_shap_np if 'Regression' not in model_name else X_test_scaled[idx_shap]

        # Si dimensions incohérentes (KernelExplainer réduit)
        if sv.shape[0] < len(X_for_plot):
            X_for_plot = X_for_plot[:sv.shape[0]]

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        fig.suptitle(f'Analyse SHAP — {model_name} ({GPU_MSG})',
                     fontsize=14, fontweight='bold')

        # ── SHAP Bar Plot (importance globale) ────────────
        plt.sca(axes[0])
        try:
            shap.summary_plot(
                sv, X_for_plot,
                feature_names=feature_names_display,
                plot_type='bar',
                max_display=min(12, len(FEATURE_COLS)),
                show=False,
                color=shap_data['color']
            )
            axes[0].set_title(f'Importance Globale des Features\n(valeurs |SHAP| moyennes)',
                              fontsize=12, fontweight='bold')
        except Exception as e:
            # Fallback : barplot manuel
            mean_shap = np.abs(sv).mean(axis=0)
            sorted_idx = np.argsort(mean_shap)[-12:]
            axes[0].barh(
                [feature_names_display[i] for i in sorted_idx],
                mean_shap[sorted_idx],
                color=shap_data['color'], alpha=0.8
            )
            axes[0].set_xlabel('|SHAP| moyen')
            axes[0].set_title('Importance Globale des Features', fontsize=12)
            print(f"  ℹ Fallback bar plot utilisé : {e}")

        # ── SHAP Dot Plot (impact directionnel) ───────────
        plt.sca(axes[1])
        try:
            shap.summary_plot(
                sv, X_for_plot,
                feature_names=feature_names_display,
                plot_type='dot',
                max_display=min(12, len(FEATURE_COLS)),
                show=False,
                cmap='RdBu'
            )
            axes[1].set_title(f'Impact Directionnel des Features\n(bleu=faible valeur, rouge=haute valeur)',
                              fontsize=12, fontweight='bold')
        except Exception as e:
            # Fallback beeswarm simplifié
            mean_shap = np.abs(sv).mean(axis=0)
            sorted_idx = np.argsort(mean_shap)[-12:]
            for j, feat_idx in enumerate(sorted_idx):
                x_vals = sv[:, feat_idx]
                y_vals = np.full_like(x_vals, j) + np.random.normal(0, 0.1, len(x_vals))
                feat_vals = X_for_plot[:, feat_idx]
                scatter = axes[1].scatter(x_vals, y_vals, c=feat_vals,
                                          cmap='RdBu', alpha=0.4, s=10)
            axes[1].set_yticks(range(len(sorted_idx)))
            axes[1].set_yticklabels([feature_names_display[i] for i in sorted_idx])
            axes[1].axvline(x=0, color='black', linewidth=0.8)
            axes[1].set_xlabel('Valeur SHAP')
            axes[1].set_title('Impact Directionnel des Features', fontsize=12)
            print(f"  ℹ Fallback dot plot utilisé : {e}")

        plt.tight_layout()
        safe_name = model_name.lower().replace(' ', '_').replace('.', '')
        plt.savefig(f'shap_{safe_name}.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"  ✓ SHAP plots sauvegardés (shap_{safe_name}.png)")

    # ── SHAP Waterfall (meilleur modèle, 1 prédiction) ────
    print(f"\n► Waterfall Plot (analyse d'une prédiction individuelle)...")

    best_shap_name = list(shap_results.keys())[0]
    for candidate in ['XGBoost', 'Random Forest', 'Lin. Regression']:
        if candidate in shap_results:
            best_shap_name = candidate
            break

    shap_data = shap_results[best_shap_name]
    sv = np.array(shap_data['values'])
    if sv.ndim == 1:
        sv = sv.reshape(1, -1)
    if isinstance(sv, list):
        sv = sv[0]

    try:
        # Choisir une prédiction "représentative" (proche de la médiane)
        y_pred_sample = results[best_shap_name]['pred_test'][idx_shap[:sv.shape[0]]]
        median_pred = np.median(y_pred_sample)
        closest_idx = np.argmin(np.abs(y_pred_sample - median_pred))

        fig, ax = plt.subplots(figsize=(12, 7))
        shap.waterfall_plot(
            shap.Explanation(
                values=sv[closest_idx],
                base_values=float(np.mean(y_train)),
                data=X_shap_np[closest_idx],
                feature_names=feature_names_display
            ),
            show=False,
            max_display=12
        )
        plt.title(f'Waterfall Plot — {best_shap_name}\n'
                  f'Prédiction individuelle (rendement prédit ≈ '
                  f'{y_pred_sample[closest_idx]:,.0f} hg/ha)',
                  fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig('shap_waterfall.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Waterfall plot sauvegardé (shap_waterfall.png)")

    except Exception as e:
        print(f"  ⚠ Waterfall plot échoué : {e}")
        # Fallback : barplot des valeurs SHAP pour un exemple
        shap_vals_single = sv[0]
        sorted_idx = np.argsort(np.abs(shap_vals_single))[-12:]
        fig, ax = plt.subplots(figsize=(10, 6))
        colors_wf = ['#f44336' if v > 0 else '#2196F3' for v in shap_vals_single[sorted_idx]]
        ax.barh([feature_names_display[i] for i in sorted_idx],
                shap_vals_single[sorted_idx],
                color=colors_wf, alpha=0.8)
        ax.axvline(x=0, color='black', linewidth=0.8)
        ax.set_xlabel('Valeur SHAP (impact sur la prédiction)')
        ax.set_title(f'Contribution des Features — {best_shap_name}\n(rouge=augmente, bleu=diminue)',
                     fontsize=12)
        plt.tight_layout()
        plt.savefig('shap_waterfall_fallback.png', dpi=100, bbox_inches='tight')
        plt.show()


In [ ]:
# ============================================================
# CELLULE 12 : RAPPORT FINAL
# ============================================================
print()
print("╔" + "═" * 58 + "╗")
print("║" + "  RAPPORT FINAL — Pipeline ML Rendement Agricole".center(58) + "║")
print("╠" + "═" * 58 + "╣")
print(f"║  Mode GPU : {GPU_MSG:<46}║")
print(f"║  Dataset  : {len(df):>6,} lignes × {len(FEATURE_COLS)} features{' '*(35-len(str(len(FEATURE_COLS))))}║")
print(f"║  Train    : {len(X_train):>6,} échantillons{' '*35}║")
print(f"║  Test     : {len(X_test):>6,} échantillons{' '*35}║")
print("╠" + "═" * 58 + "╣")
print("║  Résultats (Test Set) :".ljust(59) + "║")
print("║" + f"  {'Modèle':<20} {'RMSE':>12} {'R²':>8}".ljust(58) + "║")
print("║" + "  " + "─" * 42 + " ║")

for name, res in trained_models.items():
    marker = "🏆" if name == best_model_name else "  "
    line = f"  {marker} {name:<18} {res['rmse_test']:>12,.1f} {res['r2_test']:>8.4f}"
    print("║" + line.ljust(58) + "║")

print("╠" + "═" * 58 + "╣")
print("║  Fichiers générés :".ljust(59) + "║")
files_generated = [
    'eda_analysis.png       (Analyse Exploratoire)',
    'model_comparison.png   (Comparaison Modèles)',
]
for model_name in shap_results.keys():
    safe = model_name.lower().replace(' ', '_').replace('.', '')
    files_generated.append(f'shap_{safe}.png      (SHAP {model_name})')
files_generated.append('shap_waterfall.png     (Waterfall Plot)')

for f in files_generated:
    print(f"║  • {f:<54}║")

print("╚" + "═" * 58 + "╝")
print()
print("✅ Pipeline exécuté avec succès !")
print(f"   Top Feature (SHAP) : À lire dans les graphiques ci-dessus")


In [ ]:
# ============================================================
# CELLULE 13 (BONUS) : FEATURE IMPORTANCE INTÉGRÉE AUX MODÈLES
# ============================================================
print("=" * 60)
print("  BONUS : Feature Importance Intégrée (sans SHAP)")
print("=" * 60)
print()

fig, axes = plt.subplots(1, min(2, len(trained_models)), figsize=(16, 6))
if len(trained_models) == 1:
    axes = [axes]

plot_idx = 0

for name, res in list(trained_models.items())[:2]:
    ax = axes[plot_idx] if len(axes) > 1 else axes[0]
    model = res['model']

    try:
        # Récupérer feature importances selon le type
        if hasattr(model, 'feature_importances_'):
            importances = np.array(model.feature_importances_)
        elif hasattr(model, 'coef_'):
            importances = np.abs(np.array(model.coef_).flatten())
        else:
            raise AttributeError("Pas de feature_importances_ ni coef_")

        # Trier et afficher
        sorted_idx = np.argsort(importances)[-min(12, len(importances)):]
        feat_names_sorted = [feature_names_display[i] for i in sorted_idx]

        colors_imp = plt.cm.YlOrRd(
            np.linspace(0.3, 0.9, len(sorted_idx))
        )

        bars = ax.barh(feat_names_sorted, importances[sorted_idx],
                       color=colors_imp, edgecolor='white', linewidth=0.5)
        ax.set_xlabel("Importance relative")
        ax.set_title(f"Feature Importance\n{name}", fontsize=12, fontweight='bold')
        ax.set_xlim(0, importances[sorted_idx].max() * 1.15)

        # Ajouter les valeurs
        for bar, val in zip(bars, importances[sorted_idx]):
            ax.text(bar.get_width() + importances[sorted_idx].max() * 0.01,
                    bar.get_y() + bar.get_height()/2.,
                    f'{val:.4f}', va='center', fontsize=8)

        print(f"✓ Feature importance calculée pour {name}")
        plot_idx += 1

    except AttributeError as e:
        print(f"⚠ {name} : {e}")
        ax.text(0.5, 0.5, f'{name}\n(pas de feature importance\ndisponible)',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)
        plot_idx += 1

plt.tight_layout()
plt.savefig('feature_importance_builtin.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Feature importance intégrée sauvegardée (feature_importance_builtin.png)")


In [ ]:
# ============================================================
# CELLULE 14 : VÉRIFICATION FINALE GPU
# ============================================================
print("=" * 60)
print("  VÉRIFICATION FINALE : Utilisation GPU")
print("=" * 60)
print()

# Re-vérifier l'utilisation GPU
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        gpu_line = result.stdout.strip()
        parts = [p.strip() for p in gpu_line.split(',')]
        if len(parts) >= 4:
            print(f"GPU Modèle    : {parts[0]}")
            print(f"Utilisation   : {parts[1]}%")
            print(f"Mémoire usée  : {parts[2]} MiB / {parts[3]} MiB")
        else:
            print(f"GPU Info : {gpu_line}")
except Exception as e:
    print(f"nvidia-smi non disponible : {e}")

print()
if USE_GPU:
    print("✅ CONFIRMATION : Les modèles ont été entraînés sur GPU (RAPIDS cuML + XGBoost CUDA)")
    print(f"   • Random Forest : cuML GPU")
    print(f"   • XGBoost       : device='cuda'")
    print(f"   • Régression    : cuML GPU")
else:
    print("⚠  MODE CPU : RAPIDS non disponible, scikit-learn CPU utilisé comme fallback")
    print("   Pour activer GPU : Menu Colab > Modifier > Paramètres > GPU")

print()
print("🌾 Pipeline terminé ! Tous les graphiques sont affichés ci-dessus.")
print()
print("─" * 60)
print("Fichiers sauvegardés dans le répertoire courant :")
import glob
for f in sorted(glob.glob('*.png')):
    size = os.path.getsize(f) / 1024
    print(f"  📊 {f:<40} ({size:.0f} KB)")
print("─" * 60)
